# Empirical Risk Premia Engine v0.10 — Equity module (existing factor-pricing benchmark)

**Equity has not been abandoned.** CAPM/FF3/Carhart4/FF5/FF6, BAB, QMJ, 130 test portfolios, Fama–MacBeth identification, HAC, spanning and OOS risk attribution remain available. This packaging update embeds the **same shared ERP 0.9 source** used by the new Rates research notebook.

The new Rates v0.10 econometric experiments have not been claimed to improve Equity premia estimation. Equity next: robust FMB/traded-factor reconciliation, identification per test universe, then challenger GMM/SDF and historical conditional premia forecasting.

For the independent residual factor research, use `Equity_Residual_Factor_Research_v0_10_PORTABLE.ipynb`.


In [1]:
# Portable ERP 0.10.0 — standalone source embedded, no package install required.
from pathlib import Path
import io,zipfile,base64,sys,subprocess,importlib.util
RUNTIME=Path.cwd()/'erp_v010_runtime'
(RUNTIME/'src').mkdir(parents=True,exist_ok=True)
PAYLOAD='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as zf:
    zf.extractall(RUNTIME/'src')
for pkg in ['numpy','pandas','scipy','statsmodels','requests','openpyxl']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
sys.path.insert(0,str(RUNTIME/'src'))
for name in list(sys.modules):
    if name=='erp' or name.startswith('erp.'):
        del sys.modules[name]
import numpy as np,pandas as pd
from IPython.display import display
from erp import (__version__,EquityData,EquityPremiaEngine,analyze_equity_residuals,
                 export_snapshot,RatesData,RatesPremiaEngine,RatesEmpiricalBondPremiaEngine,
                 load_macro_panel,load_macro_series_files,analyze_rates_v09)
print('Loaded embedded ERP',__version__)


Loaded embedded ERP 0.10.0


## Configuration

Select a standard/augmented model and research universe. Set `USE_AQR = False` for a French-only offline-vendor compatibility run (`MODEL="FF6"`). If AQR disallows auto-download, upload the *original monthly* `.xlsx` files to your notebook directory and specify local paths. No long-only ETF substitution.

In [2]:
MODEL = "FF6_BAB_QMJ"    # CAPM FF3 CARHART4 FF5 FF6 FF6_BAB FF6_QMJ FF6_BAB_QMJ FF3_BAB_QMJ
UNIVERSE = "POOLED_130"   # SIZE_BM_25 INDUSTRY_30 SIZE_OP_25 SIZE_INV_25 SIZE_MOM_25 POOLED_130
AS_OF = "2026-10-31"      # strict ISO date cutoff; change when new data become available
BETA_WINDOW = 60
USE_AQR = True
AQR_PATHS = {}             # e.g. {"BAB":"/content/BAB.xlsx", "QMJ":"/content/QMJ.xlsx"}
CACHE_DIR = Path("erp_data_cache")
OUT = Path("erp_v09_equity_output")
print("Model:", MODEL, "| Universe:",UNIVERSE, "| as_of:",AS_OF)

Model: FF6_BAB_QMJ | Universe: POOLED_130 | as_of: 2026-10-31


## 1 — Official public factors and test portfolios

Downloads 25 Size×B/M, 30 Industries, 25 Size×Profitability, 25 Size×Investment and 25 Size×Momentum, plus published BAB and QMJ **monthly long/short** factors. Uses full sample intersection and preserves FF3-family vs FF5-family SMB conventions.

In [3]:
data = EquityData.from_public_sources(
    cache_dir=CACHE_DIR, aqr=USE_AQR, aqr_paths=AQR_PATHS,
)
print("Asset universes:",[(x, y.shape) for x,y in data.universes.items()])
print("AQR factors:", None if data.aqr is None else data.aqr.dropna().shape)
engine = EquityPremiaEngine(data)
f,rf = data.factors(MODEL)
display(f.loc[:AS_OF].tail())

Asset universes: [('SIZE_BM_25', (1203, 25)), ('INDUSTRY_30', (1203, 30)), ('SIZE_OP_25', (759, 25)), ('SIZE_INV_25', (759, 25)), ('SIZE_MOM_25', (1197, 25))]
AQR factors: (830, 2)


,MKT,SMB,HML,RMW,CMA,MOM,BAB,QMJ
2026-04-30,0.0995,0.0046,-0.0137,-0.0415,-0.0380,0.0951,-0.034009,-0.063683
2026-05-31,0.0491,-0.0265,-0.0231,-0.0816,-0.0146,0.0139,-0.055665,-0.075041
2026-06-30,-0.0107,0.0436,0.0358,-0.0507,0.0345,0.0462,0.022339,-0.002342
2026-07-31,-0.0061,-0.0009,0.0211,0.1107,0.0227,-0.1221,0.037362,0.142904
2026-08-31,0.0256,-0.0052,-0.0354,-0.0409,-0.0201,-0.0560,-0.003874,-0.034374


## 2 — Model horse race: identical dates

The horse race computes cross-model time-series alpha/errors on **exactly the same calendar months**. GRS and FMB diagnostics are reported separately; don't confuse lower in-sample alpha RMSE with stronger out-of-sample return forecasts.

In [4]:
MODELS = ["CAPM", "FF3", "CARHART4", "FF5", "FF6"]
if USE_AQR:
    MODELS += ["FF6_BAB", "FF6_QMJ", "FF6_BAB_QMJ"]
common_models = engine.compare_models(MODELS, UNIVERSE, as_of=AS_OF, beta_window=BETA_WINDOW)
display(common_models.sort_values("alpha_rmse_ann"))

,months,n_assets,n_factors,alpha_rmse_ann,alpha_abs_mean_ann,factor_explained_r2,sample_start,sample_end,oos_factor_rmse_monthly,oos_factor_explained_r2,oos_common_observations,oos_start,oos_end
model,,,,,,,,,,,,,
FF6,758,130,6,0.016565,0.012192,0.838563,1963-07-31,2026-08-31,0.025210,0.823292,90740,1968-07-31,2026-08-31
CARHART4,758,130,4,0.016913,0.012548,0.822282,1963-07-31,2026-08-31,0.025398,0.820651,90740,1968-07-31,2026-08-31
FF6_QMJ,758,130,7,0.017092,0.012746,0.841338,1963-07-31,2026-08-31,0.025033,0.825770,90740,1968-07-31,2026-08-31
FF6_BAB,758,130,7,0.017451,0.012910,0.840241,1963-07-31,2026-08-31,0.025368,0.821069,90740,1968-07-31,2026-08-31
FF6_BAB_QMJ,758,130,8,0.017898,0.013308,0.843016,1963-07-31,2026-08-31,0.025211,0.823283,90740,1968-07-31,2026-08-31
FF5,758,130,5,0.024937,0.017487,0.821786,1963-07-31,2026-08-31,0.026370,0.806660,90740,1968-07-31,2026-08-31
FF3,758,130,3,0.026571,0.017859,0.805297,1963-07-31,2026-08-31,0.026448,0.805510,90740,1968-07-31,2026-08-31
CAPM,758,130,1,0.028670,0.022733,0.681400,1963-07-31,2026-08-31,0.033743,0.683433,90740,1968-07-31,2026-08-31


## 3 — Fit selected factor specification, extract risks, errors and premia

Ex post factor attribution: `factor_component` is calculated from **realized current factor return**, using only beta estimated from prior months. It is not an ex-ante expected return forecast.

In [5]:
snapshot = engine.fit(MODEL, UNIVERSE, as_of=AS_OF, beta_window=BETA_WINDOW, run_fmb=True)
print("Effective data through", snapshot.as_of, "| months",snapshot.metadata['n_months'])
print("GRS (standard, if identifiable):",snapshot.metadata['grs'])
display(snapshot.fit_diagnostics.sort_values("alpha_ann").head(12))
display(snapshot.betas.head())
display(snapshot.factor_premia.to_frame("mean_excess_monthly"))
print("Static cross-sectional prices of risk (monthly) and pricing errors:")
display(snapshot.static_cs_premia.to_frame("static_CS_lambda_monthly"))
display(snapshot.static_pricing_errors.to_frame("static_CS_error_monthly").head(8))
print("Static CS diagnostics:", snapshot.metadata["static_cross_sectional_pricing"])

Effective data through 2026-08-31 00:00:00 | months 758
GRS (standard, if identifiable): {'grs_F': 2.594239956352376, 'grs_p': 6.376985397099927e-15, 'grs_note': 'standard GRS F; no covariance shrinkage'}


,alpha_monthly,alpha_ann,alpha_t_hac,adj_r2,resid_ann_vol,nobs
asset,,,,,,
INDUSTRY_30__Paper,-0.004246,-0.050956,-4.008767,0.733178,0.091098,758
INDUSTRY_30__Other,-0.003827,-0.045923,-3.510785,0.700108,0.104807,758
INDUSTRY_30__Txtls,-0.003523,-0.042280,-1.961783,0.649112,0.155391,758
INDUSTRY_30__Hshld,-0.003469,-0.041625,-3.182955,0.645884,0.095018,758
INDUSTRY_30__Books,-0.003146,-0.037748,-2.472620,0.735652,0.105209,758
INDUSTRY_30__Chems,-0.003097,-0.037168,-2.448443,0.744529,0.098082,758
INDUSTRY_30__Clths,-0.002662,-0.031946,-1.771535,0.672800,0.129974,758
INDUSTRY_30__Cnstr,-0.002486,-0.029830,-2.169741,0.809499,0.095096,758
INDUSTRY_30__Beer,-0.002384,-0.028606,-1.806538,0.489213,0.124611,758


,MKT,SMB,HML,RMW,CMA,MOM,BAB,QMJ
SIZE_BM_25__SMALL LoBM,0.977467,1.225344,-0.483896,-0.205851,-0.022434,0.000025,-0.037735,-0.435276
SIZE_BM_25__ME1 BM2,0.922065,1.217097,-0.164984,-0.368614,-0.005246,0.031216,-0.010588,-0.057995
SIZE_BM_25__ME1 BM3,0.898158,1.044540,0.089876,-0.107837,-0.011914,-0.029355,0.063257,-0.079184
SIZE_BM_25__ME1 BM4,0.898840,1.075679,0.311700,-0.138742,0.011668,0.003588,0.015698,0.105933
SIZE_BM_25__SMALL HiBM,0.917614,1.045232,0.358026,0.056911,0.227433,-0.018925,0.100659,-0.187965


,mean_excess_monthly
MKT,0.006018
SMB,0.001864
HML,0.002940
RMW,0.002514
CMA,0.002435
MOM,0.005964
BAB,0.007447
QMJ,0.003351


Static cross-sectional prices of risk (monthly) and pricing errors:


,static_CS_lambda_monthly
intercept,0.005129
MKT,0.001224
SMB,0.001854
HML,0.002396
RMW,0.001099
CMA,0.001327
MOM,0.006212
BAB,0.000395
QMJ,0.001123


,static_CS_error_monthly
SIZE_BM_25__SMALL LoBM,-0.003380
SIZE_BM_25__ME1 BM2,0.000012
SIZE_BM_25__ME1 BM3,0.000027
SIZE_BM_25__ME1 BM4,0.000686
SIZE_BM_25__SMALL HiBM,0.002021
SIZE_BM_25__ME2 BM1,-0.001083
SIZE_BM_25__ME2 BM2,0.000484
SIZE_BM_25__ME2 BM3,0.000753


Static CS diagnostics: {'cs_r2': 0.6238954200840738, 'pricing_rmse_ann': 0.014274365050202145, 'pricing_mean_abs_ann': 0.010890559152516147, 'cs_condition_raw': 42.72128743894125}


## 4 — Fama–MacBeth: prices of risk + identification quality

Lagged betas use `t-60,...,t-1`. At each month t, contemporaneous cross-sectional returns determine a realized λ_t. The table juxtaposes λ estimates (and HAC t) with cross-sectional beta dispersion, partial dispersion, auxiliary R² and scaled condition number. This is **not** Shanken-adjusted inference.

In [6]:
display(snapshot.fmb_diagnostics.round(5))
print("Historical traded-factor means vs FMB lambda estimates (monthly decimals):")
a = snapshot.factor_premia.rename("traded_factor_mean")
b = snapshot.fmb_diagnostics["lambda_monthly"].rename("FMB_lambda")
display(pd.concat([a,b],axis=1).round(5))
print("Last 5 monthly lambda observations:")
display(snapshot.fmb_lambdas.tail().round(4))
if USE_AQR:
    print("Official-factor spanning vs FF6 (not a time-series trading signal):")
    display(pd.DataFrame([engine.spanning_test(target="BAB", against_model="FF6", as_of=AS_OF),
                          engine.spanning_test(target="QMJ", against_model="FF6", as_of=AS_OF)]))

,lambda_monthly,lambda_ann_arithmetic,t_hac,n_months,n_assets,beta_std,beta_q05,beta_q95,partial_beta_std,auxiliary_r2,cs_rank,cs_n_columns,cs_condition_scaled
factor,,,,,,,,,,,,,
BAB,0.00068,0.00818,0.55088,698,130.0,0.36893,-0.41058,0.88809,0.29063,0.37945,9.0,9.0,2.33345
CMA,0.00134,0.01603,1.51736,698,130.0,0.29275,-0.29975,0.58026,0.24865,0.27863,9.0,9.0,2.33345
HML,0.00265,0.03174,2.00945,698,130.0,0.28889,-0.31950,0.62328,0.27318,0.10584,9.0,9.0,2.33345
MKT,-0.00167,-0.02009,-0.89519,698,130.0,0.16586,0.64376,1.19513,0.14104,0.27685,9.0,9.0,2.33345
MOM,0.00462,0.05548,3.14737,698,130.0,0.20987,-0.36678,0.30942,0.19854,0.10505,9.0,9.0,2.33345
QMJ,0.00052,0.00626,0.53167,698,130.0,0.29620,-0.64079,0.30730,0.23946,0.34644,9.0,9.0,2.33345
RMW,0.00048,0.00570,0.57784,698,130.0,0.31902,-0.53585,0.50665,0.26676,0.30082,9.0,9.0,2.33345
SMB,0.00106,0.01276,0.91690,698,130.0,0.47540,-0.19656,1.17462,0.45621,0.07913,9.0,9.0,2.33345
intercept,0.00780,0.09359,4.86480,698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Historical traded-factor means vs FMB lambda estimates (monthly decimals):


,traded_factor_mean,FMB_lambda
MKT,0.00602,-0.00167
SMB,0.00186,0.00106
HML,0.00294,0.00265
RMW,0.00251,0.00048
CMA,0.00243,0.00134
MOM,0.00596,0.00462
BAB,0.00745,0.00068
QMJ,0.00335,0.00052
intercept,NaN,0.00780


Last 5 monthly lambda observations:


,intercept,MKT,SMB,HML,RMW,CMA,MOM,BAB,QMJ
2026-04-30,-0.0254,0.1132,0.0113,-0.0228,-0.0415,-0.0392,0.0663,-0.0332,-0.0849
2026-05-31,-0.0061,0.0493,-0.0158,-0.0210,-0.0405,-0.0163,0.0206,-0.0439,-0.0589
2026-06-30,-0.0310,0.0369,0.0451,0.0385,0.0067,0.0008,0.0825,0.0304,0.0003
2026-07-31,0.1085,-0.1246,-0.0056,0.0435,0.0434,-0.0053,-0.0928,0.0170,0.0814
2026-08-31,0.1402,-0.1199,-0.0054,-0.0061,-0.0608,-0.0014,-0.0761,-0.0695,-0.0636


Official-factor spanning vs FF6 (not a time-series trading signal):


,target,against_model,n_months,alpha_monthly,alpha_ann,alpha_t_hac,r2
0,BAB,FF6,758,0.003111,0.037329,2.292079,0.267309
1,QMJ,FF6,758,0.002865,0.034382,5.272185,0.695519


## 5 — Risk exposure / alpha / residual decomposition

`alpha` is the in-sample regression intercept. `residual_returns` are idiosyncratic regression residuals. They do **not** establish a monetizable future alpha.

In [7]:
print("Alpha (annualized, first 10 assets)")
display(snapshot.alpha.head(10).mul(12).to_frame("fitted_alpha_ann"))
print("Realized factor-attribution sample (beta known strictly before each observation)")
display(snapshot.oos_attribution.tail(12))
print("Fitted residual covariance (first 5 assets)")
display(snapshot.residuals.iloc[:,-5:].cov().round(5))

Alpha (annualized, first 10 assets)


,fitted_alpha_ann
SIZE_BM_25__SMALL LoBM,-0.013608
SIZE_BM_25__ME1 BM2,0.018443
SIZE_BM_25__ME1 BM3,0.008164
SIZE_BM_25__ME1 BM4,0.013962
SIZE_BM_25__SMALL HiBM,0.023012
SIZE_BM_25__ME2 BM1,-0.001533
SIZE_BM_25__ME2 BM2,0.008354
SIZE_BM_25__ME2 BM3,0.007985
SIZE_BM_25__ME2 BM4,-0.000943
SIZE_BM_25__ME2 BM5,-0.001698


Realized factor-attribution sample (beta known strictly before each observation)


factor_component  realized_excess_return  \
Date       asset                                                              
2026-08-31 SIZE_OP_25__ME3 OP4             0.003363               -0.001470   
           SIZE_OP_25__ME3 OP5             0.005275                0.009876   
           SIZE_OP_25__ME4 OP1             0.062590                0.074410   
           SIZE_OP_25__ME4 OP2             0.015314               -0.001263   
           SIZE_OP_25__ME4 OP3             0.021553               -0.007483   
           SIZE_OP_25__ME4 OP4             0.013227                0.022070   
           SIZE_OP_25__ME4 OP5             0.012725                0.005610   
           SIZE_OP_25__ME5 OP2             0.028202                0.058078   
           SIZE_OP_25__ME5 OP3             0.018348                0.040872   
           SIZE_OP_25__ME5 OP4             0.023362               -0.008670   
           SIZE_OP_25__SMALL HiOP         -0.002591               -0.013380   
           SIZE_OP_25__SMALL LoOP          0.056184                0.081303   

                                   unexplained_including_alpha  
Date       asset                                                
2026-08-31 SIZE_OP_25__ME3 OP4                       -0.004833  
           SIZE_OP_25__ME3 OP5                        0.004601  
           SIZE_OP_25__ME4 OP1                        0.011820  
           SIZE_OP_25__ME4 OP2                       -0.016577  
           SIZE_OP_25__ME4 OP3                       -0.029036  
           SIZE_OP_25__ME4 OP4                        0.008843  
           SIZE_OP_25__ME4 OP5                       -0.007115  
           SIZE_OP_25__ME5 OP2                        0.029876  
           SIZE_OP_25__ME5 OP3                        0.022524  
           SIZE_OP_25__ME5 OP4                       -0.032032  
           SIZE_OP_25__SMALL HiOP                    -0.010789  
           SIZE_OP_25__SMALL LoOP                     0.025119

Fitted residual covariance (first 5 assets)


,SIZE_MOM_25__BIG LoPRIOR,SIZE_MOM_25__ME5 PRIOR2,SIZE_MOM_25__ME5 PRIOR3,SIZE_MOM_25__ME5 PRIOR4,SIZE_MOM_25__BIG HiPRIOR
SIZE_MOM_25__BIG LoPRIOR,0.00067,0.00002,-0.00004,-0.00003,0.00002
SIZE_MOM_25__ME5 PRIOR2,0.00002,0.00026,0.00004,0.00002,-0.00003
SIZE_MOM_25__ME5 PRIOR3,-0.00004,0.00004,0.00020,0.00004,-0.00003
SIZE_MOM_25__ME5 PRIOR4,-0.00003,0.00002,0.00004,0.00020,-0.00000
SIZE_MOM_25__BIG HiPRIOR,0.00002,-0.00003,-0.00003,-0.00000,0.00025


## 6 — Export to common file schema

Exports CSV tables plus `manifest.json` and explicit meanings/limitations, consumable by Slicer/Referee without modifying their own statistical logic.

In [8]:
destination = export_snapshot(snapshot, OUT / (UNIVERSE + "__" + MODEL))
print("Exported:",destination)
print("Files:",sorted(x.name for x in destination.iterdir()))
from erp.integrations import to_alpha_referee_factor_library
print("Alpha Referee factor library:", list(to_alpha_referee_factor_library(snapshot)["factors"].columns))
# Alpha Slicer v0.5.1 integration, only if you have actual strategy & benchmark time series:
# from erp.integrations import to_alpha_slicer_inputs
# slicer_kwargs = to_alpha_slicer_inputs(snapshot, strategy_net=real_net,
#                                       strategy_gross=real_gross, benchmark=real_benchmark)
# slicer_result = your_existing_slicer.fit(**slicer_kwargs)

Exported: erp_v09_equity_output/POOLED_130__FF6_BAB_QMJ
Files: ['alpha.csv', 'asset_excess_returns.csv', 'betas.csv', 'factor_premia.csv', 'factor_returns.csv', 'fit_diagnostics.csv', 'fmb_diagnostics.csv', 'fmb_lambdas.csv', 'manifest.json', 'oos_attribution.csv', 'residual_returns.csv', 'static_cs_premia.csv', 'static_pricing_errors.csv']
Alpha Referee factor library: ['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'MOM', 'BAB', 'QMJ']


## 7 — Research interpretation

Interpret each result separately: **factor fit**, **cross-sectional price of risk**, **identification strength**, **spanning**, **OOS attribution** and **predictive usefulness** are different questions. Re-run the same dates for comparisons with published QMJ/BAB; do not infer a universal best model. The historical v0.1 notebook remains unchanged and is included in the source ZIP as provenance.